In [0]:
SELECT * FROM bronze.orders LIMIT 500;

In [0]:
SELECT * FROM bronze.orders WHERE order_id IS NULL;

In [0]:
-- 1. Tabela Silver de Pedidos (tratamento de datas e cálculo de atraso)
-- CAST atua como um conversor de valor em formato especificado; utilizado aqui para garantir formato de dados correto mesmo com reprocessamento dos CSVs
CREATE OR REPLACE TABLE silver.orders AS
SELECT
  order_id, -- Colunas inalteradas permanecerão com o nome original em inglês
  customer_id,
  order_status,
  CAST(order_purchase_timestamp AS TIMESTAMP) AS data_compra, -- TIMESTAMP é mais adequado aqui e mais leve
  CAST(order_delivered_customer_date AS TIMESTAMP) AS data_entrega_real,
  CAST(order_estimated_delivery_date AS TIMESTAMP) AS data_entrega_estimada,
  DATEDIFF(
    CAST(order_delivered_customer_date AS DATE),
    CAST(order_estimated_delivery_date AS DATE)
  ) AS dias_diferenca_entrega,
  CASE
    WHEN order_delivered_customer_date IS NULL THEN 'Não Entregue'
    WHEN CAST(order_delivered_customer_date AS DATE) > CAST(order_estimated_delivery_date AS DATE) THEN 'Atrasado'
    ELSE 'No Prazo'
  END AS status_sla
FROM bronze.orders
WHERE order_id IS NOT NULL;

SELECT * FROM silver.orders LIMIT 500;

In [0]:
SELECT * FROM bronze.order_items LIMIT 500; 

In [0]:
-- 2. Tabela Silver de Itens do Pedido
CREATE OR REPLACE TABLE silver.order_items AS
SELECT
  order_id,
  order_item_id,
  product_id,
  seller_id,
  CAST(price AS DECIMAL(10,2)) AS valor_produto,
  CAST(freight_value AS DECIMAL(10,2)) AS valor_frete
FROM bronze.order_items;

SELECT * FROM silver.order_items LIMIT 500;

In [0]:
SELECT * FROM bronze.products LIMIT 500; 

In [0]:
-- 3. Tabela Silver de Produtos
CREATE OR REPLACE TABLE silver.products AS
SELECT
  product_id,
  COALESCE(REPLACE(product_category_name, '_', ' '), 'Sem categoria') AS categoria_produto
FROM bronze.products;

SELECT * FROM silver.products LIMIT 500;

In [0]:
SELECT * FROM bronze.customers LIMIT 500; 

In [0]:
-- 4. Tabela Silver de Clientes
CREATE OR REPLACE TABLE silver.customers AS
SELECT
  customer_id,
  customer_unique_id,
  INITCAP(customer_city) AS cidade,
  UPPER(customer_state) AS estado
FROM bronze.customers;

SELECT * FROM silver.customers LIMIT 500;

In [0]:
SELECT * FROM bronze.order_reviews LIMIT 500; 

In [0]:
-- 5. Seleção de 300 avaliações com comentário escrito para utilizar na IA 
CREATE OR REPLACE TABLE silver.reviews_amostra_ia AS
SELECT
  review_id,
  order_id,
  CAST(review_score AS INT) AS nota_avaliacao,
  TRIM(review_comment_message) AS comentario,
  CAST(review_creation_date AS TIMESTAMP) AS data_avaliacao
FROM bronze.order_reviews
WHERE review_comment_message IS NOT NULL
  AND LENGTH(TRIM(review_comment_message)) >= 15
LIMIT 300;

SELECT * FROM silver.reviews_amostra_ia;